In [1]:
#conda activate burnseverity
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import requests
import json

import geopandas as gpd

import rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from rasterio import features
from rasterio.plot import show_hist

from shapely.geometry import shape, mapping
from shapely.ops import unary_union

import validation as val

from src import config
from src import process


### Validation goals

There a four **questions** we want to test:
1. Do our boundaries and those from CalFire match?
2. How do the two different indices (dNBR and RBR) calculated with our tool differ?
3. How does the time window chosed affect the resultts
4. For fires, where this is available,how does the dNBR maps from our tool compare to those from BAER?

We will use the following **metrics** to test these questions
1. Percent overlap between Calfire shapefile and vectorized pixels with $x > 0$
2. Compare  mean, spread, SD, and interquartile range dNBR and RBR
3. Compare mean and variance for all time windows tested and the two reference points (alarm date vs. containment date)
4. $R^2$ on a pixel basis between the two images 

This notebook calculates these metrics where they require **post-processing tasks**
1. Filter for pixels with $x > 0$ and convert to a shapefile, calculate overlap with Calfire boundary
2. & 3. Calculate mean, median, SD, spread, and interquartile range of pixels within Calfire boundaries
4. Recoarse Sentinel to Landsat, calculate $R^2$ based on Calfire boundary

In [2]:
fires = pd.read_csv(config.PATH_JOBS_LOG)
calfire = gpd.read_file(config.PATH_FIRES_VALIDATION)
fire_names = calfire['FIRE_NAME'].unique()

In [3]:
indicators = []

for fire_name in fire_names:
  
    calfire_polygon = calfire[calfire['FIRE_NAME'] == fire_name]

    for metric in config.METRICS:

        print(f'Processing fire: {fire_name}, {metric} ...')

        for post_fire_reference_point in config.POST_FIRE_REFERENCE_POINT:

            for post_fire_period in config.POST_FIRE_PERIOD:

                fire_event_name = fires.loc[(fires['fire_name'] == fire_name) & 
                                (fires['post_fire_days'] == post_fire_period) &
                                (fires['date_mode'] == post_fire_reference_point), 'fire_event_name'].values[0]

                url_raster = process.get_url_raster(fires, fire_name, post_fire_period, post_fire_reference_point, metric)
                raster = process.get_raster_as_lonlat(url_raster)[0]
                polygon_from_raster = process.convert_burnscar_to_polygon(raster)
                polygon_from_raster.to_file(f'{config.PATH_SHP}{fire_event_name}_from_{metric}.shp')

                mean, var, max, min, q_25, q_50, q_75 = process.calculate_statistical_indicators(raster, calfire_polygon)

                process.append_results(fire_name, post_fire_reference_point, post_fire_period, metric, 
                                       mean, var, max, min, q_25, q_50, q_75, indicators)
                

df_indicators = pd.DataFrame(indicators)
df_indicators.to_csv(f'output/{config.PATH_INDICATORS}', index=False)


                



Processing fire: COFFEE POT, dnbr ...
Processing fire: COFFEE POT, rbr ...
Processing fire: SENTINEL, dnbr ...
Processing fire: SENTINEL, rbr ...
Processing fire: SIMPSON, dnbr ...
Processing fire: SIMPSON, rbr ...
Processing fire: YORK, dnbr ...


AttributeError: 'NoneType' object has no attribute 'get'